# Visualisation du Nettoyage des Données (Avant / Après)

Ce notebook utilise `pandas` pour illustrer concrètement l'impact de l'étape de nettoyage automatique sur nos événements OpenAgenda.
Nous allons comparer les données brutes (`events_raw.json`) avec les données propres prêtes pour le RAG (`events_clean.json`).

In [ ]:
import pandas as pd
import json

# 1. Chargement des données brutes (Avant nettoyage)
with open('../data/events_raw.json', 'r', encoding='utf-8') as f:
    df_raw = pd.DataFrame(json.load(f))

# 2. Chargement des données propres (Après nettoyage)
with open('../data/events_clean.json', 'r', encoding='utf-8') as f:
    df_clean = pd.DataFrame(json.load(f))

print("Données chargées avec succès pour la comparaison.")

## 1. Bilan Général : Doublons et Valeurs Manquantes
Le tableau suivant compare le volume de données, la présence de doublons (basé sur l'identifiant `uid`) et le nombre d'événements sans adresse.

In [ ]:
# Calcul des statistiques comparatives
stats_data = {
    "Étape": ["Avant Nettoyage (Brut)", "Après Nettoyage (Propre)"],
    "Nombre total d'événements": [len(df_raw), len(df_clean)],
    "Doublons (UID)": [df_raw.duplicated(subset=['uid']).sum(), df_clean.duplicated(subset=['uid']).sum()],
    "Sans Localisation (Valeurs manquantes)": [df_raw['location'].isnull().sum(), df_clean['location'].isnull().sum()]
}

df_stats = pd.DataFrame(stats_data).set_index("Étape")

# Affichage visuel sous forme de tableau Pandas
df_stats

## 2. Valeurs incorrectes ou bruitées : Le problème du multilingue
Dans les données brutes, les champs textuels (titre, description) sont des dictionnaires contenant plusieurs langues (français, anglais, espagnol...). 
Cela constitue des **données incorrectes** ou bruitées pour notre RAG qui ne fonctionne qu'en français. 

Voici la comparaison visuelle d'un champ avant et après extraction de la version française :

In [ ]:
print("\n❌ AVANT NETTOYAGE (Champ 'title' complet avec toutes les langues) :")
print(json.dumps(df_raw['title'].iloc[0], indent=2, ensure_ascii=False))

print("\n" + "-"*50)

print("\n✅ APRÈS NETTOYAGE (Champ 'title_fr' ciblé) :")
print(df_clean['title_fr'].iloc[0])

## 3. Aperçu visuel des données prêtes pour l'indexation FAISS
Voici à quoi ressemblent les premières lignes de notre DataFrame final, propre, unifié et prêt à être injecté dans notre base vectorielle.

In [ ]:
cols_to_display = ['uid', 'title_fr', 'description_fr', 'location']
df_clean[cols_to_display].head()